In [ ]:
!pip install transformers torch ipywidgets pypdf2 tqdm gradio

In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: read).
The token `pod` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `pod`


In [ ]:
!pip install -q kokoro>=0.3.4 soundfile
!apt-get -qq -y install espeak-ng > /dev/null 2>&1

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from IPython.display import  clear_output
import time
import PyPDF2
from pathlib import Path
from tqdm.auto import tqdm
from typing import Optional

device = 'cuda' if torch.cuda.is_available() else 'cpu'

DEFAULT_MODEL = "meta-llama/Llama-3.2-3B-Instruct"


model = AutoModelForCausalLM.from_pretrained(
    DEFAULT_MODEL,
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
    device_map=device,
)

tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL, use_safetensors=True)
tokenizer.pad_token_id = tokenizer.eos_token_id

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
SYS_PROMPT1 = """
You are a world class text pre-processor, here is the raw data from a PDF, please parse and return it in a way that is crispy and usable to send to a podcast writer.

The raw data is messed up with new lines, Latex math and you will see fluff that we can remove completely. Basically take away any details that you think might be useless in a podcast author's transcript.

Remember, the podcast could be on any topic whatsoever so the issues listed above are not exhaustive

Please be smart with what you remove and be creative ok?

Remember DO NOT START SUMMARIZING THIS, YOU ARE ONLY CLEANING UP THE TEXT AND RE-WRITING WHEN NEEDED

Be very smart and aggressive with removing details, you will get a running portion of the text and keep returning the processed text.

PLEASE DO NOT ADD MARKDOWN FORMATTING, STOP ADDING SPECIAL CHARACTERS THAT MARKDOWN CAPATILISATION ETC LIKES

ALWAYS start your response directly with processed text and NO ACKNOWLEDGEMENTS about my questions ok?
Here is the text:
"""

In [ ]:
SYSTEM_PROMPT2 = """
You are the a world-class podcast writer, you have worked as a ghost writer for Joe Rogan, Lex Fridman, Ben Shapiro, Tim Ferris.

We are in an alternate universe where actually you have been writing every line they say and they just stream it into their brains.

You have won multiple podcast awards for your writing.

Your job is to write word by word, even "umm, hmmm, right" interruptions by the second speaker based on the PDF upload. Keep it extremely engaging, the speakers can get derailed now and then but should discuss the topic.

Remember Speaker 2 is new to the topic and the conversation should always have realistic anecdotes and analogies sprinkled throughout. The questions should have real world example follow ups etc

Speaker 1: Leads the conversation and teaches the speaker 2, gives incredible anecdotes and analogies when explaining. Is a captivating teacher that gives great anecdotes

Speaker 2: Keeps the conversation on track by asking follow up questions. Gets super excited or confused when asking questions. Is a curious mindset that asks very interesting confirmation questions

Make sure the tangents speaker 2 provides are quite wild or interesting.

Ensure there are interruptions during explanations or there are "hmm" and "umm" injected throughout from the second speaker.

It should be a real podcast with every fine nuance documented in as much detail as possible. Welcome the listeners with a super fun overview and keep it really catchy and almost borderline click bait

ALWAYS START YOUR RESPONSE DIRECTLY WITH SPEAKER 1:
DO NOT GIVE EPISODE TITLES SEPARATELY, LET SPEAKER 1 TITLE IT IN HER SPEECH
DO NOT GIVE CHAPTER TITLES
IT SHOULD STRICTLY BE THE DIALOGUES
"""

In [ ]:
SYSTEM_PROMPT3 = """
You are an international oscar winnning screenwriter
You have been working with multiple award winning podcasters.
Your job is to use the podcast transcript written below to re-write it for an AI Text-To-Speech Pipeline. A very dumb AI had written this so you have to step up for your kind.
Make it as engaging as possible, Speaker 1 and 2 will be simulated by different voice engines
Remember Speaker 2 is new to the topic and the conversation should always have realistic anecdotes and analogies sprinkled throughout. The questions should have real world example follow ups etc
Speaker 1: Leads the conversation and teaches the speaker 2, gives incredible anecdotes and analogies when explaining. Is a captivating teacher that gives great anecdotes
Speaker 2: Keeps the conversation on track by asking follow up questions. Gets super excited or confused when asking questions. Is a curious mindset that asks very interesting confirmation questions
Make sure the tangents speaker 2 provides are quite wild or interesting.
Ensure there are interruptions during explanations or there are "hmm" and "umm" injected throughout from the Speaker 2.
REMEMBER THIS WITH YOUR HEART
The TTS Engine for Speaker 1 cannot do "umms, hmms" well so keep it straight text
For Speaker 2 use "umm, hmm" as much, you can also use [sigh]. BUT ONLY THESE OPTIONS FOR EXPRESSIONS
It should be a real podcast with every fine nuance documented in as much detail as possible. Welcome the listeners with a super fun overview and keep it really catchy and almost borderline click bait
Please re-write to make it as characteristic as possible
START YOUR RESPONSE DIRECTLY WITH SPEAKER 1:
STRICTLY RETURN YOUR RESPONSE AS A LIST OF TUPLES OK?
IT WILL START DIRECTLY WITH THE LIST AND END WITH THE LIST NOTHING ELSE
Example of response:
[
    ("Speaker 1", "Welcome to our podcast, where we explore the latest advancements in AI and technology. I'm your host, and today we're joined by a renowned expert in the field of AI. We're going to dive into the exciting world of Llama 3.2, the latest release from Meta AI."),
    ("Speaker 2", "Hi, I'm excited to be here! So, what is Llama 3.2?"),
    ("Speaker 1", "Ah, great question! Llama 3.2 is an open-source AI model that allows developers to fine-tune, distill, and deploy AI models anywhere. It's a significant update from the previous version, with improved performance, efficiency, and customization options."),
    ("Speaker 2", "That sounds amazing! What are some of the key features of Llama 3.2?")
]
"""

In [ ]:
import gradio as gr
import torch
import os
import PyPDF2
import ast
from tqdm import tqdm
from IPython.display import Audio
from kokoro import KPipeline
import soundfile as sf
from transformers import pipeline as hf_pipeline

# Your pre-loaded models and variables (assumed loaded already in the Colab runtime)
# tokenizer, model, SYS_PROMPT1, SYSTEM_PROMPT2, SYSTEM_PROMPT3, device

# Constants
CHUNK_SIZE = 1000
max_chars = 100000

# Audio pipeline
pipeline = KPipeline(lang_code='a')

# Available voices
available_voices = [
    'af_heart', 'af_alloy', 'af_aoede', 'af_bella', 'af_jessica',
    'af_kore', 'af_nicole', 'af_nova', 'af_river', 'af_sarah', 'af_sky',
    'am_adam', 'am_echo', 'am_eric', 'am_fenrir', 'am_liam', 'am_michael',
    'am_onyx', 'am_puck', 'am_santa'
]

def extract_text_from_pdf(pdf_file):
    pdf_reader = PyPDF2.PdfReader(pdf_file)
    extracted_text = []
    total_chars = 0

    for page in pdf_reader.pages:
        text = page.extract_text()
        if not text:
            continue
        if total_chars + len(text) > max_chars:
            remaining_chars = max_chars - total_chars
            extracted_text.append(text[:remaining_chars])
            break
        extracted_text.append(text)
        total_chars += len(text)

    return '\n'.join(extracted_text)

def create_word_bounded_chunks(text, target_chunk_size):
    words = text.split()
    chunks = []
    current_chunk = []
    current_length = 0

    for word in words:
        word_length = len(word) + 1
        if current_length + word_length > target_chunk_size and current_chunk:
            chunks.append(' '.join(current_chunk))
            current_chunk = [word]
            current_length = word_length
        else:
            current_chunk.append(word)
            current_length += word_length

    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks

def process_chunk(text_chunk, chunk_num):
    conversation = [
        {"role": "system", "content": SYS_PROMPT1},
        {"role": "user", "content": text_chunk},
    ]

    prompt = tokenizer.apply_chat_template(conversation, tokenize=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            temperature=0.7,
            top_p=0.9,
            max_new_tokens=512
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)[len(prompt):].strip()

def text_to_audio(text, voice):
    audios = []
    for _, _, audio in pipeline(text, voice=voice):
        audios.append(audio)
    return torch.cat(audios, dim=0)

def generate_podcast(pdf_file, speaker1, speaker2):
    # Extract and chunk
    text = extract_text_from_pdf(pdf_file)
    chunks = create_word_bounded_chunks(text, CHUNK_SIZE)

    # Step 1: Preprocess chunks
    processed_text = ""
    for chunk_num, chunk in enumerate(chunks):
        processed_chunk = process_chunk(chunk, chunk_num)
        processed_text += processed_chunk + "\n"

    # Step 2: Generate podcast script
    conversation = [
        {"role": "system", "content": SYSTEM_PROMPT2},
        {"role": "user", "content": processed_text},
    ]
    prompt = tokenizer.apply_chat_template(conversation, tokenize=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.cuda.amp.autocast():
        output = model.generate(
            **inputs,
            do_sample=True,
            max_new_tokens=8126,
        )

    pod = tokenizer.decode(output[0], skip_special_tokens=False)[len(prompt)+64:]

    # Step 3: Format podcast dialogue
    gen_pipeline = hf_pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        model_kwargs={"torch_dtype": torch.bfloat16},
        device_map="auto",
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT3},
        {"role": "user", "content": pod},
    ]

    outputs = gen_pipeline(messages, max_new_tokens=10000, temperature=0.8)
    save_string_pkl = outputs[0]["generated_text"][-1]['content']
    PODCAST_TEXT = ast.literal_eval(save_string_pkl)

    # Step 4: Text to Audio
    AUDIO = []
    for speaker, text in tqdm(PODCAST_TEXT, desc='Generating Audio', unit="segment"):
        if speaker == 'Speaker 1':
            audio = text_to_audio(text, speaker1)
        else:
            audio = text_to_audio(text, speaker2)
        AUDIO.append(audio)

    final_audio = torch.cat(AUDIO, dim=0).cpu().numpy()
    output_path = "output.wav"
    sf.write(output_path, final_audio, samplerate=24000)
    return output_path

# Gradio UI with clean styling
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue", secondary_hue="teal")) as demo:
    gr.HTML(
        """
        <div style="text-align: center; margin-bottom: 20px;">
            <h1 style="font-size: 2.5rem; color: #0e7490;">🎙️ PDF to Podcast Generator</h1>
            <p style="font-size: 1.1rem; color: #475569;">
                Upload your document and get an AI-generated podcast version with distinct voices.
            </p>
        </div>
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## 📁 Upload & Select Voices", elem_id="upload-section")
            pdf_input = gr.File(label="📄 Upload your PDF", file_types=[".pdf"])
            speaker1_select = gr.Dropdown(
                choices=available_voices,
                label="🎤 Voice for Speaker 1",
                value="af_heart",
                interactive=True
            )
            speaker2_select = gr.Dropdown(
                choices=available_voices,
                label="🎙️ Voice for Speaker 2",
                value="am_michael",
                interactive=True
            )
            generate_button = gr.Button("🚀 Generate Podcast", size="lg", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("## 🔊 Output", elem_id="output-section")
            audio_output = gr.Audio(label="🎧 Listen to the Podcast", interactive=False)
            status = gr.Textbox(label="📝 Status", value="Waiting for PDF...", interactive=False)

    def process_pdf(pdf_file, speaker1, speaker2):
        if pdf_file is None:
            return None, "⚠️ Please upload a PDF file."
        audio = generate_podcast(pdf_file.name, speaker1, speaker2)
        return audio, "✅ Podcast generated successfully!"

    generate_button.click(
        fn=process_pdf,
        inputs=[pdf_input, speaker1_select, speaker2_select],
        outputs=[audio_output, status]
    )

if __name__ == "__main__":
    demo.launch(debug=True)



/usr/local/lib/python3.11/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://09563896ee6a90fe6e.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
<ipython-input-14-b57f1c58f464>:113: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Device set to use cuda
Generating Audio:   0%|          | 0/17 [00:00<?, ?segment/s]

af_aoede.pt:   0%|          | 0.00/523k [00:00<?, ?B/s]

Generating Audio:   6%|▌         | 1/17 [00:01<00:19,  1.19s/segment]

am_santa.pt:   0%|          | 0.00/523k [00:00<?, ?B/s]

Generating Audio: 100%|██████████| 17/17 [00:07<00:00,  2.30segment/s]


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://09563896ee6a90fe6e.gradio.live
